# Stage 1 M Signal Training on Colab

This notebook runs the current Stage 1 idiom-signal models on Colab and downloads only their output folders.

## Prepare the project archive

From the repo root on your local machine, create a lightweight zip with only the Stage 1 code, LLM results, and requirements:

```bash
zip -r stage1_idiom_signal_colab.zip stage1_idiom_signal llm_results requirements.txt \
  -x "stage1_idiom_signal/*/outputs/*" "*/__pycache__/*" ".DS_Store" "*/.DS_Store"
```

Upload `stage1_idiom_signal_colab.zip` in the next cell.

In [ ]:
%cd /content

from google.colab import files

uploaded = files.upload()
archive_name = "stage1_idiom_signal_colab.zip"
assert archive_name in uploaded, f"Upload {archive_name}"

In [ ]:
%cd /content
!rm -rf /content/nlpproject
!mkdir -p /content/nlpproject
!unzip -q /content/stage1_idiom_signal_colab.zip -d /content/nlpproject
%cd /content/nlpproject

## Install dependencies

Install the same project dependencies inside the Colab runtime.

In [ ]:
!pip install -q transformers sentencepiece sentence-transformers pillow scikit-learn tqdm pandas matplotlib seaborn optuna

## Build matched LLM records

This creates `stage1_idiom_signal/data/matched_records.json`, which is required by the perturbation and fusion models.

In [ ]:
!python stage1_idiom_signal/prepare_data.py

## Train all Stage 1 models

Each model writes only its best weights, config, and metrics under `stage1_idiom_signal/<model>/outputs/`.

In [ ]:
!python stage1_idiom_signal/run_all_models.py

## Read model summary

This table checks the aggregate metrics saved by each model.

In [ ]:
import json
from pathlib import Path

import pandas as pd

models = ["baseline", "marked_simple", "marked_mlp", "perturbation_mlp", "fusion_mlp"]
rows = []

for model_name in models:
    path = Path("stage1_idiom_signal") / model_name / "outputs" / "metrics.json"
    data = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "model": model_name,
        "mean_val_f1": data["mean_val_f1"],
        "best_seed": data["best_seed"],
        "best_val_f1": data["best_val_f1"],
        "best_epoch": data["best_epoch"],
    })

summary = pd.DataFrame(rows).sort_values("mean_val_f1", ascending=False)
display(summary)

## Download outputs

The zip keeps repo-relative paths, so unzip it at the local repo root to place files back into the correct folders.

In [ ]:
!rm -f stage1_idiom_signal_outputs.zip
!zip -qr stage1_idiom_signal_outputs.zip \
  stage1_idiom_signal/baseline/outputs \
  stage1_idiom_signal/marked_simple/outputs \
  stage1_idiom_signal/marked_mlp/outputs \
  stage1_idiom_signal/perturbation_mlp/outputs \
  stage1_idiom_signal/fusion_mlp/outputs \
  stage1_idiom_signal/data/matched_records.json

files.download("stage1_idiom_signal_outputs.zip")